In [ ]:
import utils
import numpy as np
import pandas as pd
import os
import glob
import base64
from pathlib import Path
from IPython.display import HTML, display
from check_inclusion_files import filepaths

import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_formats = ['svg']
plt.style.use('seaborn-v0_8')

#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_columns', 20)
#pd.set_option('display.width', 150)

In [ ]:
unique_subject_ids = [48]
included_subjects_list = [39, 48, 49, 52, 53, 54, 55, 57, 58, 60, 61, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 76, 77, 78, 79, 81, 82, 83, 85, 86, 87, 88, 92, 94, 95]

In [ ]:
def show_svg(path, width='100%', autoplay=False):
    """Embed an SVG file in the notebook output.

    fMRIPrep flicker reportlets (reference <-> fieldmap, before <-> after) are
    inlined so the hover animation works; other SVGs go in as an <img>.
    autoplay=True makes flicker reportlets loop without hovering.
    """
    path = Path(path)
    text = path.read_text()
    # FIX: flicker reportlets need inline markup, because <img> ignores hover
    if 'foreground-svg' in text:
        markup = text[text.find('<svg'):]  # drop any XML prolog or DOCTYPE
        if autoplay:
            markup = markup.replace('infinite paused', 'infinite running', 1)
        display(HTML(f'<div style="width:{width}">{markup}</div>'))
    else:
        data = base64.b64encode(path.read_bytes()).decode()
        display(HTML(f'<img src="data:image/svg+xml;base64,{data}" style="width:{width}">'))

# Is all data stored and processed correctly?

In [ ]:
def check_files_exist(file_paths_list: list, label: str = 'files') -> list:
    """
    Report which files or folders in a list are missing, without raising an error.

    Each entry can be a plain path or a glob pattern (*, ?, ** for any depth).
    A pattern counts as found if at least one path matches it. Prints to stdout,
    so the report ends up in the HTML summary.

    Parameters
    ----------
    file_paths_list : list of str
        Paths or glob patterns to check.
    label : str
        Name for the group of paths, used in the printed report.

    Returns
    -------
    list of str
        The paths or patterns with no match. Empty if all were found.
    """
    missing_files_list = []
    for file_path in file_paths_list:
        try:
            path_found = len(glob.glob(file_path, recursive=True)) > 0  # CHANGED: was os.path.isfile
        except (TypeError, OSError):
            path_found = False
        if not path_found:
            missing_files_list.append(file_path)

    number_found_int = len(file_paths_list) - len(missing_files_list)
    print(f'{number_found_int} of {len(file_paths_list)} {label} found.')
    if missing_files_list:
        print(f'Missing {label}:')
        for file_path in missing_files_list:
            print(f'  - {file_path}')

    return missing_files_list

In [ ]:
files_to_check = [filepath.format(subject_id=unique_subject_ids[0]) for filepath in filepaths]

check_files_exist(files_to_check)

# Behavioural checks

In [ ]:
# Preparation
plot_range = [0, 6]

# Loads the datapath, subject_id's and session numbers
raw_output_path = '/Volumes/project/3025011.02/raw/'
unique_sessions = [2,3,4]

# Load and combine the datasets
combined_results_dataframe = utils.import_functions.main(raw_output_path, unique_subject_ids, unique_sessions, remove_reversals=True)

In [ ]:
# Multiply some variables by 100 to show percentage correct etc
combined_results_dataframe['objectively_correct_boolean'] = combined_results_dataframe['objectively_correct_boolean'] * 100
mapping = {'approach': 1, 'avoid': 0}
combined_results_dataframe['response_boolean'] = combined_results_dataframe['response'].map(mapping)
combined_results_dataframe['response_boolean'] = combined_results_dataframe['response_boolean'].astype('Int64')
combined_results_dataframe_shortened = combined_results_dataframe[combined_results_dataframe['trials_since_reversal_combined_learning']<=10]

In [ ]:
# Re-import data with the invalid trials to check RT effects
combined_results_dataframe_reaction_time = utils.import_functions.main(raw_output_path, unique_subject_ids, unique_sessions, remove_invalid_trials=False)
combined_results_dataframe_reaction_time['RT_ms'] = combined_results_dataframe_reaction_time['RT_s'] * 1000

## Session 2

In [ ]:
combined_results_dataframe_session = combined_results_dataframe[combined_results_dataframe['session']==2]

### Check whether they get tired

In [ ]:
combined_results_dataframe_session['objectively_correct_rolling_average'] = combined_results_dataframe_session['objectively_correct_boolean'].rolling(window=100).mean()

utils.plotting_functions.lineplot_se(
    df=combined_results_dataframe_session,
    x_col='trial', y_col='objectively_correct_rolling_average',
    xlabel_name='trial [n]', ylabel_name='accuracy [%]',
    subject_col='subject_id',
    legend_outside=True,
    trend=['quadratic'], show_trend_stats=True, trend_colour='red',
    plot_name='rolling average of accuracy over time',
    save=False
);

### Performance during session 2

In [ ]:
# Jitter vertically to make the lines easier to read when overlapping
dataframe = combined_results_dataframe_session.dropna(subset=['response'])
dataframe = dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'])
dataframe['trial_separated'] = (
    dataframe
    .groupby(['subject_id', 'session', 'stimuli_type'])
    .cumcount()
    + 1
)
dataframe = utils.analysis_functions.replace_strings_with_integers(dataframe,
                                                                                    {'avoid': 100, 'approach': 0},
                                                                                    'response',
                                                                                    'response_int')
dataframe['response_rolling_average'] = (
    dataframe
    .groupby('stimuli_type')['response_int']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

stimuli_list = ['Angry', 'Happy']
colour_list = ['red', 'green']

stim_types = dataframe['stimuli_type'].unique()
n_types = len(stim_types)

# create one subplot per stimuli_type
fig, axes = plt.subplots(n_types, 1,
                         figsize=(15, 4 * n_types),
                         sharex=True)

# if there's only one type, axes is not a list
if n_types == 1:
    axes = [axes]

for ax, stim in zip(axes, stim_types):
    sub = dataframe[dataframe['stimuli_type'] == stim]
    ax.plot(sub['trial_separated'], sub['probability_condition'], linestyle='-',
            label=f'{stimuli_list[stim-1]} probability condition', color=colour_list[stim-1])
    ax.plot(sub['trial_separated'], sub['response_rolling_average'],
            marker='s', linestyle='--',
            label='Response rolling average', color='#5688C7')
    ax.set_title(f'{stimuli_list[stim-1]} stimuli')
    ax.set_ylabel('Chance of avoid response [%]')
    ax.legend(loc='best')

# common x-axis label
axes[-1].set_xlabel('Trial')

plt.tight_layout()
plt.show()

### Check reaction times for non-responders

In [ ]:
combined_results_dataframe_reaction_time_session = combined_results_dataframe_reaction_time[combined_results_dataframe_reaction_time['session']==2]

In [ ]:
# Print percentage of non-response trials
non_response_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['response'].isna()]
percentage_responses = len(non_response_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_responses > 10:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
      'This means that the participant responded in less than 90% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
          'Include the participant.\n')

joystick_locked_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['RT_s'] < 0.05]
percentage_joystick_locked = len(joystick_locked_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_joystick_locked > 10:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
      'This means that the participant locked the joystick in more than 10% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
          'Include the participant.\n')

In [ ]:
combined_results_pivoted = (
    combined_results_dataframe_reaction_time_session
    .groupby('response')['RT_ms']
    .apply(lambda s: s.reset_index(drop=True))
    .unstack(level=0)
    .rename_axis(None, axis=1)
    .reset_index(drop=True)
)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    combined_results_pivoted,
    columns=['approach', 'avoid'],
    labels=['Approach', 'Avoid'],
    flip=['left','right'],
    ylabel_name='Reaction time (ms)',
    plot_name='Response effect on reaction time',
    connect_pairs=False,
    save=False
);

## Session 3

In [ ]:
combined_results_dataframe_session = combined_results_dataframe[combined_results_dataframe['session']==3]

### Check whether they get tired

In [ ]:
combined_results_dataframe_session['objectively_correct_rolling_average'] = combined_results_dataframe_session['objectively_correct_boolean'].rolling(window=100).mean()

utils.plotting_functions.lineplot_se(
    df=combined_results_dataframe_session,
    x_col='trial', y_col='objectively_correct_rolling_average',
    xlabel_name='trial [n]', ylabel_name='accuracy [%]',
    subject_col='subject_id',
    legend_outside=True,
    trend=['quadratic'], show_trend_stats=True, trend_colour='red',
    plot_name='rolling average of accuracy over time',
    save=False
);

### Performance during session 3

In [ ]:
# Jitter vertically to make the lines easier to read when overlapping
dataframe = combined_results_dataframe_session.dropna(subset=['response'])
dataframe = dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'])
dataframe['trial_separated'] = (
    dataframe
    .groupby(['subject_id', 'session', 'stimuli_type'])
    .cumcount()
    + 1
)
dataframe = utils.analysis_functions.replace_strings_with_integers(dataframe,
                                                                                    {'avoid': 100, 'approach': 0},
                                                                                    'response',
                                                                                    'response_int')
dataframe['response_rolling_average'] = (
    dataframe
    .groupby('stimuli_type')['response_int']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

stimuli_list = ['Angry', 'Happy']
colour_list = ['red', 'green']

stim_types = dataframe['stimuli_type'].unique()
n_types = len(stim_types)

# create one subplot per stimuli_type
fig, axes = plt.subplots(n_types, 1,
                         figsize=(15, 4 * n_types),
                         sharex=True)

# if there's only one type, axes is not a list
if n_types == 1:
    axes = [axes]

for ax, stim in zip(axes, stim_types):
    sub = dataframe[dataframe['stimuli_type'] == stim]
    ax.plot(sub['trial_separated'], sub['probability_condition'], linestyle='-',
            label=f'{stimuli_list[stim-1]} probability condition', color=colour_list[stim-1])
    ax.plot(sub['trial_separated'], sub['response_rolling_average'],
            marker='s', linestyle='--',
            label='Response rolling average', color='#5688C7')
    ax.set_title(f'{stimuli_list[stim-1]} stimuli')
    ax.set_ylabel('Chance of avoid response [%]')
    ax.legend(loc='best')

# common x-axis label
axes[-1].set_xlabel('Trial')

plt.tight_layout()
plt.show()

### Check reaction times for non-responders

In [ ]:
combined_results_dataframe_reaction_time_session = combined_results_dataframe_reaction_time[combined_results_dataframe_reaction_time['session']==3]

In [ ]:
# Print percentage of non-response trials
non_response_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['response'].isna()]
percentage_responses = len(non_response_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_responses > 10:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
      'This means that the participant responded in less than 90% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
          'Include the participant.\n')

joystick_locked_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['RT_s'] < 0.05]
percentage_joystick_locked = len(joystick_locked_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_joystick_locked > 10:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
      'This means that the participant locked the joystick in more than 10% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
          'Include the participant.\n')

In [ ]:
combined_results_pivoted = (
    combined_results_dataframe_reaction_time_session
    .groupby('response')['RT_ms']
    .apply(lambda s: s.reset_index(drop=True))
    .unstack(level=0)
    .rename_axis(None, axis=1)
    .reset_index(drop=True)
)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    combined_results_pivoted,
    columns=['approach', 'avoid'],
    labels=['Approach', 'Avoid'],
    flip=['left','right'],
    ylabel_name='Reaction time (ms)',
    plot_name='Response effect on reaction time',
    connect_pairs=False,
    save=False
);

## Session 4

In [ ]:
combined_results_dataframe_session = combined_results_dataframe[combined_results_dataframe['session']==4]

### Check whether they get tired

In [ ]:
combined_results_dataframe_session['objectively_correct_rolling_average'] = combined_results_dataframe_session['objectively_correct_boolean'].rolling(window=100).mean()

utils.plotting_functions.lineplot_se(
    df=combined_results_dataframe_session,
    x_col='trial', y_col='objectively_correct_rolling_average',
    xlabel_name='trial [n]', ylabel_name='accuracy [%]',
    subject_col='subject_id',
    legend_outside=True,
    trend=['quadratic'], show_trend_stats=True, trend_colour='red',
    plot_name='rolling average of accuracy over time',
    save=False
);

### Performance during session 4

In [ ]:
# Jitter vertically to make the lines easier to read when overlapping
dataframe = combined_results_dataframe_session.dropna(subset=['response'])
dataframe = dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'])
dataframe['trial_separated'] = (
    dataframe
    .groupby(['subject_id', 'session', 'stimuli_type'])
    .cumcount()
    + 1
)
dataframe = utils.analysis_functions.replace_strings_with_integers(dataframe,
                                                                                    {'avoid': 100, 'approach': 0},
                                                                                    'response',
                                                                                    'response_int')
dataframe['response_rolling_average'] = (
    dataframe
    .groupby('stimuli_type')['response_int']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean())
)

stimuli_list = ['Angry', 'Happy']
colour_list = ['red', 'green']

stim_types = dataframe['stimuli_type'].unique()
n_types = len(stim_types)

# create one subplot per stimuli_type
fig, axes = plt.subplots(n_types, 1,
                         figsize=(15, 4 * n_types),
                         sharex=True)

# if there's only one type, axes is not a list
if n_types == 1:
    axes = [axes]

for ax, stim in zip(axes, stim_types):
    sub = dataframe[dataframe['stimuli_type'] == stim]
    ax.plot(sub['trial_separated'], sub['probability_condition'], linestyle='-',
            label=f'{stimuli_list[stim-1]} probability condition', color=colour_list[stim-1])
    ax.plot(sub['trial_separated'], sub['response_rolling_average'],
            marker='s', linestyle='--',
            label='Response rolling average', color='#5688C7')
    ax.set_title(f'{stimuli_list[stim-1]} stimuli')
    ax.set_ylabel('Chance of avoid response [%]')
    ax.legend(loc='best')

# common x-axis label
axes[-1].set_xlabel('Trial')

plt.tight_layout()
plt.show()

### Check reaction times for non-responders

In [ ]:
combined_results_dataframe_reaction_time_session = combined_results_dataframe_reaction_time[combined_results_dataframe_reaction_time['session']==4]

In [ ]:
# Print percentage of non-response trials
non_response_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['response'].isna()]
percentage_responses = len(non_response_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_responses > 10:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
      'This means that the participant responded in less than 90% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'{len(non_response_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials are non-response trials\n'
      f'So, {percentage_responses:.2f}% of the trials are non-response trials\n'
          'Include the participant.\n')

joystick_locked_trials = combined_results_dataframe_reaction_time_session[combined_results_dataframe_reaction_time_session['RT_s'] < 0.05]
percentage_joystick_locked = len(joystick_locked_trials)/len(combined_results_dataframe_reaction_time_session)*100

if percentage_joystick_locked > 10:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
      'This means that the participant locked the joystick in more than 10% of the trials.\n'
          'Exclude the participant.\n')
else:
    print(f'Participant locked the joystick in {len(joystick_locked_trials)} out of {len(combined_results_dataframe_reaction_time_session)} trials\n'
      f"That's {percentage_joystick_locked:.2f}% of the trials\n"
          'Include the participant.\n')

In [ ]:
combined_results_pivoted = (
    combined_results_dataframe_reaction_time_session
    .groupby('response')['RT_ms']
    .apply(lambda s: s.reset_index(drop=True))
    .unstack(level=0)
    .rename_axis(None, axis=1)
    .reset_index(drop=True)
)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    combined_results_pivoted,
    columns=['approach', 'avoid'],
    labels=['Approach', 'Avoid'],
    flip=['left','right'],
    ylabel_name='Reaction time (ms)',
    plot_name='Response effect on reaction time',
    connect_pairs=False,
    save=False
);

## Check the congruency effect

In [ ]:
# Single-pass aggregation & reshape
combined_results_pivoted = (
    combined_results_dataframe
    .pivot_table(
        index=['session', 'valence'],
        columns='congruency',
        values='objectively_correct_boolean',
        aggfunc='mean'
    )
    .reset_index()
)
combined_results_pivoted = combined_results_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    combined_results_pivoted,
    columns=['congruent','incongruent'],
    labels=['Congruent','Incongruent'],
    flip=['left','right'],
    overlay_by='valence',
    legend_title='Valence',
    connect_pairs=[(0, 1)],
    colours=['red','green'],
    ylabel_name='Accuracy [%]',
    plot_name='Congruency effect on accuracy',
    aspect_ratio=4/3,
    save=False
);

In [ ]:
combined_results_dataframe_shortened['response_valence'] = combined_results_dataframe_shortened['most_rewarding_response'].astype(str) + '_' + combined_results_dataframe_shortened['valence'].astype(str)
print(combined_results_dataframe_shortened[['trials_since_reversal_combined_learning','response_boolean','response_valence','session']])

utils.plotting_functions.lineplot_se(
    df=combined_results_dataframe_shortened,
    x_col='trials_since_reversal_combined_learning', y_col='response_boolean',
    xlabel_name='trials since reversal', ylabel_name='p(approach)',
    subject_col='session', condition_col='response_valence',
    legend_outside=True,
    plot_name='Congruency effect on accuracy after reversals',
    save=False
);

In [ ]:
# Single-pass aggregation & reshape
combined_results_pivoted = (
    combined_results_dataframe
    .pivot_table(
        index=['session','valence'],
        columns=['volatility', 'congruency'],
        values='objectively_correct_boolean',
        aggfunc='mean'
    )
    .reset_index()
)
combined_results_pivoted.columns = [
    '_'.join(str(x) for x in col if x)
    for col in combined_results_pivoted.columns
]

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    combined_results_pivoted,
    columns=['stable_congruent','stable_incongruent','volatile_congruent','volatile_incongruent'],
    labels=['Congruent','Incongruent','Congruent','Incongruent'],
    group_labels=[
        [('Stable', [0, 1]), ('Volatile', [2, 3])]
    ],
    flip=['left','right','left','right'],
    connect_pairs=[(0, 1), (2, 3)],
    colours=['red','green'],
    ylabel_name='Accuracy [%]',
    overlay_by='valence',
    legend_title='Valence',
    plot_name='Congruency effect on accuracy split for volatility',
    save=False
);

# Stimulation efficacy

In [ ]:
simulation_results_dataframe = '/Volumes/project/3025011.02/TUS_simulations/simulation_results.csv'
simulation_results_dataframe = pd.read_csv(simulation_results_dataframe)
simulation_results_dataframe = simulation_results_dataframe[simulation_results_dataframe['subject']==f'sub-{unique_subject_ids[0]:03d}']

In [ ]:
simulation_results_dataframe_filtered = simulation_results_dataframe.loc[simulation_results_dataframe['target'].str.contains('target_1')]

# Single-pass aggregation & reshape
simulation_results_pivoted = (
    simulation_results_dataframe_filtered
    .pivot_table(
        index=['subject', 'target'],
        columns='condition',
        values='mean_Ipa',
        aggfunc='mean'
    )
    .reset_index()
)
simulation_results_pivoted = simulation_results_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    simulation_results_pivoted,
    columns=['planning','post-hoc'],
    flip=['left','right'],
    colours=['lightcoral','brown'],
    ylabel_name='Mean Ipa in Amygdala [W/cm2]',
    plot_name='Ipa in Amygdala',
    font_scale=1.6,
    save=False
);

In [ ]:
simulation_results_dataframe_filtered = simulation_results_dataframe.loc[simulation_results_dataframe['target'].str.contains('target_1')]

# Single-pass aggregation & reshape
simulation_results_pivoted = (
    simulation_results_dataframe_filtered
    .pivot_table(
        index=['subject', 'target'],
        columns='condition',
        values='Isppa',
        aggfunc='mean'
    )
    .reset_index()
)
simulation_results_pivoted = simulation_results_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    simulation_results_pivoted,
    columns=['planning','post-hoc'],
    flip=['left','right'],
    colours=['lightcoral','brown'],
    ylabel_name='Isppa in Amygdala [W/cm2]',
    plot_name='Isppa in Amygdala',
    font_scale=1.6,
    save=False
);

In [ ]:
try:
    display(utils.plotting_functions.view_nifti_slices([
        {'path': f'/Volumes/project/3025011.02/bids/sub-{unique_subject_ids[0]:03d}/ses-mri01/anat/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-mprage_T1w.nii.gz'},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/segmentation_data/m2m_sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_amygdala_left.nii.gz', 'colormap': 'green', 'opacity': 0.8},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/segmentation_data/m2m_sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_amygdala_right.nii.gz', 'colormap': 'green', 'opacity': 0.8},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_1_1/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_1_1_dc20_strength_2.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 1, 'cal_max': 8},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_1_2/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_1_2_dc20_strength_2.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 1, 'cal_max': 8},
    ]))
except FileNotFoundError as e:
    print(f'No viewer, file missing: {e.filename}')

In [ ]:
simulation_results_dataframe_filtered = simulation_results_dataframe.loc[simulation_results_dataframe['target'].str.contains('target_2')]

# Single-pass aggregation & reshape
simulation_results_pivoted = (
    simulation_results_dataframe_filtered
    .pivot_table(
        index=['subject', 'target'],
        columns='condition',
        values='mean_Ipa',
        aggfunc='mean'
    )
    .reset_index()
)
simulation_results_pivoted = simulation_results_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    simulation_results_pivoted,
    columns=['planning','post-hoc'],
    flip=['left','right'],
    colours=['steelblue','midnightblue'],
    ylabel_name='Mean Ipa in dACC [W/cm2]',
    plot_name='Ipa in dacc',
    font_scale=1.6,
    save=False
);

In [ ]:
simulation_results_dataframe_filtered = simulation_results_dataframe.loc[simulation_results_dataframe['target'].str.contains('target_2')]

# Single-pass aggregation & reshape
simulation_results_pivoted = (
    simulation_results_dataframe_filtered
    .pivot_table(
        index=['subject', 'target'],
        columns='condition',
        values='Isppa',
        aggfunc='mean'
    )
    .reset_index()
)
simulation_results_pivoted = simulation_results_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    simulation_results_pivoted,
    columns=['planning','post-hoc'],
    flip=['left','right'],
    colours=['steelblue','midnightblue'],
    ylabel_name='Isppa in dACC [W/cm2]',
    plot_name='Isppa in dACC',
    font_scale=1.6,
    save=False
);

In [ ]:
try:
    display(utils.plotting_functions.view_nifti_slices([
        {'path': f'/Volumes/project/3025011.02/bids/sub-{unique_subject_ids[0]:03d}/ses-mri01/anat/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-mprage_T1w.nii.gz'},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/segmentation_data/m2m_sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_payam_dacc_left_mask.nii.gz', 'colormap': 'green', 'opacity': 0.8},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/segmentation_data/m2m_sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_payam_dacc_right_mask.nii.gz', 'colormap': 'green', 'opacity': 0.8},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_1/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_1_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_2/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_2_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_3/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_3_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_4/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_4_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_5/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_5_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
        {'path': f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/target_2_6/sub-{unique_subject_ids[0]:03d}/sub-{unique_subject_ids[0]:03d}_final_isppa_orig_coord_target_2_6_dc20_strength_3.nii.gz', 'colormap': 'hot', 'opacity': 0.8, 'cal_min': 3, 'cal_max': 10},
    ]))
except FileNotFoundError as e:
    print(f'No viewer, file missing: {e.filename}')

# Stimulation safety

In [ ]:
simulation_directory = f'/Volumes/project/3025011.02/TUS_simulations/post-hoc/'

stimulation_output_full_filenames_1 = os.path.join(simulation_directory, '**', '*layered_output_table_target_1*.csv')
stimulation_output_full_filenames_2 = os.path.join(simulation_directory, '**', '*layered_output_table_target_2*.csv')
stimulation_output_full_filenames_3 = os.path.join(simulation_directory, '**', '*layered_output_table_target_3*.csv')

stimulation_output_full_filenames = glob.glob(stimulation_output_full_filenames_1, recursive=True) + glob.glob(stimulation_output_full_filenames_2, recursive=True) + glob.glob(stimulation_output_full_filenames_3, recursive=True)

In [ ]:
# List to hold individual dataframes
list_of_dataframes = []

# Process each CSV file
for stimulation_output_full_filename in stimulation_output_full_filenames:
    # Extract the filename (without the path)
    stimulation_output_filename = os.path.basename(stimulation_output_full_filename)
    # Remove the '.csv' extension
    stimulation_output_filename_without_extension = os.path.splitext(stimulation_output_filename)[0]

    # Split the filename by underscores
    stimulation_output_filename_parts = stimulation_output_filename_without_extension.split('_')

    # Extract the desired parts from the filename:
    subject_id = stimulation_output_filename_parts[0]                     # e.g. 'sub-010'
    target = '_'.join(stimulation_output_filename_parts[4:7])               # e.g. 'target_2_2'

    # Read the CSV file (assuming it contains one row)
    try:
        df = pd.read_csv(stimulation_output_full_filename)
    except Exception as e:
        print(f"Error reading {stimulation_output_full_filename}: {e}")
        continue  # Skip this file if there's an error

    # Add the extracted metadata as new columns
    df['target'] = target

    # Append the dataframe to our list
    list_of_dataframes.append(df)

# Combine all dataframes into one
if list_of_dataframes:
    stimulation_output = pd.concat(list_of_dataframes, ignore_index=True)
    print(stimulation_output)
else:
    raise Exception("No CSV files were found or processed.")

In [ ]:
stimulation_output = stimulation_output[stimulation_output['subject_id']==unique_subject_ids[0]]

In [ ]:
expected = [f"target_{i}_{j}"
            for i in range(1, 4)
            for j in range(1, 7)]

# find which expected names are missing
missing = [t for t in expected
           if not any(t in filename for filename in stimulation_output_full_filenames)]

if missing:
    print("Missing target names:")
    for name in missing:
        print("  -", name)
    simulations_unfinished = True
else:
    print("All target names 1_1 through 3_6 are present.")
    simulations_unfinished = False

In [ ]:
# Reorder columns
column_indices = list(range(stimulation_output.shape[1]))
reordered_column_indices = [column_indices[0]] + column_indices[-3:] + column_indices[1:-3]
stimulation_output = stimulation_output.iloc[:, reordered_column_indices]
stimulation_output.sort_values(by=['subject_id', 'target'], inplace=True)
stimulation_output.reset_index(drop=True, inplace=True)

# Now drop some columns
stimulation_output.drop(
    ['real_focal_distance','Ix_brain', 'Iy_brain','Iz_brain',
     'focus_pos_final_1', 'focus_pos_final_2', 'focus_pos_final_3',
     'trans_pos_final_1', 'trans_pos_final_2', 'trans_pos_final_3'],
    axis=1,
    inplace=True
)

# Split target up into two
stimulation_output[['main_target', 'sub_target']] = stimulation_output['target'].str.extract(r'(target_\d+)_(\d+)')

print(stimulation_output)

In [ ]:
# Single-pass aggregation & reshape
stimulation_output_pivoted = (
    stimulation_output
    .pivot_table(
        index=['subject_id','sub_target'],
        columns='main_target',
        values='max_MI_brain',
        aggfunc='mean'
    )
    .reset_index()
)
stimulation_output_pivoted = stimulation_output_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    stimulation_output_pivoted,
    columns=['target_1', 'target_2', 'target_3'],
    labels=['Amygdala', 'dACC', 'Defocused'],
    colours=['firebrick', 'royalblue','darkgrey'],
    ylabel_name='MItc',
    hline=1.9,
    plot_name='Maximum MItc',
    font_scale=1.6,
    save=False
);

In [ ]:
# Single-pass aggregation & reshape
stimulation_output_pivoted = (
    stimulation_output
    .pivot_table(
        index=['subject_id','sub_target'],
        columns='main_target',
        values='CEM43_skin',
        aggfunc='mean'
    )
    .reset_index()
)
stimulation_output_pivoted = stimulation_output_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    stimulation_output_pivoted,
    columns=['target_1', 'target_2', 'target_3'],
    labels=['Amygdala', 'dACC', 'Defocused'],
    colours=['firebrick', 'royalblue','darkgrey'],
    ylabel_name='CEM43 skin [min]',
    hline=21,
    plot_name='Maximum CEM43 skin',
    logarithmic_scale=True,
    font_scale=1.6,
    save=False
);

In [ ]:
# Single-pass aggregation & reshape
stimulation_output_pivoted = (
    stimulation_output
    .pivot_table(
        index=['subject_id','sub_target'],
        columns='main_target',
        values='CEM43_brain',
        aggfunc='mean'
    )
    .reset_index()
)
stimulation_output_pivoted = stimulation_output_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    stimulation_output_pivoted,
    columns=['target_1', 'target_2', 'target_3'],
    labels=['Amygdala', 'dACC', 'Defocused'],
    colours=['firebrick', 'royalblue','darkgrey'],
    ylabel_name='CEM43 brain [min]',
    hline=2,
    plot_name='Maximum CEM43 brain',
    logarithmic_scale=True,
    font_scale=1.6,
    save=False
);

In [ ]:
# Single-pass aggregation & reshape
stimulation_output_pivoted = (
    stimulation_output
    .pivot_table(
        index=['subject_id','sub_target'],
        columns='main_target',
        values='CEM43_skull',
        aggfunc='mean'
    )
    .reset_index()
)
stimulation_output_pivoted = stimulation_output_pivoted.rename_axis(None, axis=1)

# Plot the two groups using a raincloud plot
utils.plotting_functions.paired_raincloud(
    stimulation_output_pivoted,
    columns=['target_1', 'target_2', 'target_3'],
    labels=['Amygdala', 'dACC', 'Defocused'],
    colours=['firebrick', 'royalblue','darkgrey'],
    ylabel_name='CEM43 skull [min]',
    hline=16,
    plot_name='Maximum CEM43 skull',
    logarithmic_scale=True,
    font_scale=1.6,
    save=False
);

# fMRI checks

## Anatomical

In [ ]:
# Load the raw joystick data of one subject and session, average it per volume and add the fMRIPrep confounds of that volume
def load_joystick_with_confounds(subject_id, session, project_folder='/Volumes/project/3025011.02/', confound_columns=None):
    joystick_files = utils.import_functions.list_files_with_date_and_subject_id(
        os.path.join(project_folder, 'raw'), [subject_id], [session], selected_pattern='joystick_output')
    joystick_file = joystick_files[0]  # The list function already raises an error if no file is found

    # The trigger file of the same run sits in the same folder and shares the joystick file's suffix
    joystick_basename = os.path.basename(joystick_file)
    trigger_file = os.path.join(os.path.dirname(joystick_file),
                                joystick_basename.replace('joystick_output_', 'trigger_signals_', 1))
    confounds_file = (f'{project_folder}/bids/derivatives/fmriprep/sub-{subject_id:03}/ses-mri{session:02}/func/'
                      f'sub-{subject_id:03}_ses-mri{session:02}_task-AARL_desc-confounds_timeseries.tsv')
    if not os.path.exists(confounds_file):
        raise FileNotFoundError(f'fMRIprep has not yet been run for subject {subject_id} and session {session}')

    joystick = pd.read_csv(joystick_file, sep=';')
    trigger_signals = pd.read_csv(trigger_file, sep=';', index_col=0)
    confounds = pd.read_csv(confounds_file, sep='\t', na_values='n/a')
    if confound_columns is not None:
        confounds = confounds[confound_columns]
    confounds = confounds.fillna(0)  # Same as create_design_matrix.ipynb, removes the n/a values in volume 0

    # Every volume needs exactly one trigger, otherwise all confounds shift
    if len(trigger_signals) != len(confounds):
        raise ValueError(f'sub-{subject_id:03} session-{session:02}: {len(trigger_signals)} triggers '
                         f'but {len(confounds)} confound rows')
    if not (trigger_signals['trigger'].to_numpy() == np.arange(1, len(trigger_signals) + 1)).all():
        raise ValueError(f'sub-{subject_id:03} session-{session:02}: trigger numbers are not sequential from 1')

    # Trigger 1 is the first row of the confounds file
    volumes = pd.DataFrame({'volume': np.arange(len(confounds)),
                            'volume_onset': trigger_signals['trigger_time'].to_numpy(dtype=float)})
    confounds = pd.concat([volumes, confounds.reset_index(drop=True)], axis=1)

    # Assign each sample to the last volume onset at or before it, no further than one TR (plus jitter)
    repetition_time = np.median(np.diff(confounds['volume_onset']))
    joystick = pd.merge_asof(joystick.sort_values('timepoint'), confounds[['volume', 'volume_onset']],
                             left_on='timepoint', right_on='volume_onset',
                             direction='backward', tolerance=repetition_time * 1.1)

    # One row per volume: the confounds are constant within a volume, so average the joystick samples first
    joystick = (joystick.dropna(subset=['volume'])
                .astype({'volume': int})
                .groupby('volume', as_index=False)
                .agg(trial=('trial', 'first'), n_samples=('location_y', 'size'), location_y=('location_y', 'mean')))
    return joystick.merge(confounds, on='volume', how='left'), confounds

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_space-MNI152NLin6Asym_T1w.svg')

#### T1w

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/mriqc/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-mprage_desc-background_T1w.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/mriqc/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-mprage_desc-segmentation_T1w.svg')

#### T2w

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/mriqc/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-sagisop2ellscan_desc-background_T2w.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/mriqc/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri01_acq-sagisop2ellscan_desc-segmentation_T2w.svg')

## Session 2
### Cue > Baseline contrast

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-carpetplot_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-confoundcorr_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-compcorvar_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_acq-fieldmap_fmapid-fieldmapsesmri02_desc-phasediff_fieldmap.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-coreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-fmapCoreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri02_task-AARL_desc-sdc_bold.svg')

In [ ]:
try:
    display(utils.plotting_functions.view_nifti_slices([
        {'path': f'/Volumes/kenvdzee/Documents/phd_1_analysis/analysis_fMRI/MNI152_T1_1mm_brain.nii.gz'},
        {'path': f'/Volumes/project/3025011.02/bids/derivatives/fsl/sub-{unique_subject_ids[0]:03d}/ses-mri02/func/sub-{unique_subject_ids[0]:03d}_ses-mri02_baseline_confounds.feat/stats/zstat1.nii.gz', 'colormap': 'hot', 'colormapNegative': 'winter', 'opacity': 0.4, 'cal_min': 2.3, 'cal_max': 10},
    ]))
except FileNotFoundError as e:
    print(f'No viewer, file missing: {e.filename}')

### Check movement

In [ ]:
joystick_confounds_dataframe, confounds_dataframe = load_joystick_with_confounds(unique_subject_ids[0], unique_sessions[0])

In [ ]:
mean_fd_cutoff = 0.4
mean_fd_cutoff = 1
max_fd_cutoff = 5

In [ ]:
mean_fd = confounds_dataframe['framewise_displacement'].mean()
if mean_fd >= mean_fd_cutoff:
    print(f'Mean FD of {mean_fd}mm => {mean_fd_cutoff}mm')

fd_proportion_exceeding_cutoff = (confounds_dataframe['framewise_displacement'] > mean_fd_cutoff).mean()
fd_proportion_cutoff = 0.2
if fd_proportion_exceeding_cutoff >= fd_proportion_cutoff:
    print(f'{fd_proportion_exceeding_cutoff:.1%} of volumes have FD > {mean_fd_cutoff}mm, >= {fd_proportion_cutoff:.0%}')

max_fd = confounds_dataframe['framewise_displacement'].max()
if max_fd >= max_fd_cutoff:
    print(f'Max FD of {max_fd}mm => {max_fd_cutoff}mm')

### Relate joystick movements to confound regressors

In [ ]:
 joystick_movement_correlation_threshold = 0.3

In [ ]:
# Correlate the joystick coordinate with FD and the six motion parameters, one value per volume
motion_columns = ['framewise_displacement', 'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
correlations = joystick_confounds_dataframe[motion_columns].corrwith(joystick_confounds_dataframe['location_y'], method='spearman')
moderate_correlations = correlations[correlations.abs() >=  joystick_movement_correlation_threshold]

if not moderate_correlations.empty:
    print(f'Spearman\'s rho correlation between confound regressors and average joystick location within a volume.\n'
    f'Consists of {len(joystick_confounds_dataframe)} volumes with joystick samples.\n')
    print(moderate_correlations.round(3).rename('spearman rho').to_string())

### Check overall movement

## Session 3
### Cue > Baseline contrast

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-carpetplot_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-confoundcorr_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-compcorvar_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_acq-fieldmap_fmapid-fieldmapsesmri03_desc-phasediff_fieldmap.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-coreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-fmapCoreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri03_task-AARL_desc-sdc_bold.svg')

In [ ]:
try:
    display(utils.plotting_functions.view_nifti_slices([
        {'path': f'/Volumes/kenvdzee/Documents/phd_1_analysis/analysis_fMRI/MNI152_T1_1mm_brain.nii.gz'},
        {'path': f'/Volumes/project/3025011.02/bids/derivatives/fsl/sub-{unique_subject_ids[0]:03d}/ses-mri03/func/sub-{unique_subject_ids[0]:03d}_ses-mri03_baseline_confounds.feat/stats/zstat1.nii.gz', 'colormap': 'hot', 'colormapNegative': 'winter', 'opacity': 0.4, 'cal_min': 2.3, 'cal_max': 10},
    ]))
except FileNotFoundError as e:
    print(f'No viewer, file missing: {e.filename}')

### Check movement

In [ ]:
joystick_confounds_dataframe, confounds_dataframe = load_joystick_with_confounds(unique_subject_ids[0], unique_sessions[1])

In [ ]:
mean_fd = confounds_dataframe['framewise_displacement'].mean()
if mean_fd >= mean_fd_cutoff:
    print(f'Mean FD of {mean_fd}mm => {mean_fd_cutoff}mm')

fd_proportion_exceeding_cutoff = (confounds_dataframe['framewise_displacement'] > mean_fd_cutoff).mean()
fd_proportion_cutoff = 0.2
if fd_proportion_exceeding_cutoff >= fd_proportion_cutoff:
    print(f'{fd_proportion_exceeding_cutoff:.1%} of volumes have FD > {mean_fd_cutoff}mm, >= {fd_proportion_cutoff:.0%}')

max_fd = confounds_dataframe['framewise_displacement'].max()
if max_fd >= max_fd_cutoff:
    print(f'Max FD of {max_fd}mm => {max_fd_cutoff}mm')

### Relate joystick movements to confound regressors

In [ ]:
# Correlate the joystick coordinate with FD and the six motion parameters, one value per volume
motion_columns = ['framewise_displacement', 'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
correlations = joystick_confounds_dataframe[motion_columns].corrwith(joystick_confounds_dataframe['location_y'], method='spearman')
moderate_correlations = correlations[correlations.abs() >=  joystick_movement_correlation_threshold]

if not moderate_correlations.empty:
    print(f'Spearman\'s rho correlation between confound regressors and average joystick location within a volume.\n'
    f'Consists of {len(joystick_confounds_dataframe)} volumes with joystick samples.\n')
    print(moderate_correlations.round(3).rename('spearman rho').to_string())

## Session 4
### Cue > Baseline contrast

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-carpetplot_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-confoundcorr_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-compcorvar_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_acq-fieldmap_fmapid-fieldmapsesmri04_desc-phasediff_fieldmap.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-coreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-fmapCoreg_bold.svg')

In [ ]:
show_svg(f'/Volumes/project/3025011.02/bids/derivatives/fmriprep/sub-{unique_subject_ids[0]:03d}/figures/sub-{unique_subject_ids[0]:03d}_ses-mri04_task-AARL_desc-sdc_bold.svg')

In [ ]:
try:
    display(utils.plotting_functions.view_nifti_slices([
        {'path': f'/Volumes/kenvdzee/Documents/phd_1_analysis/analysis_fMRI/MNI152_T1_1mm_brain.nii.gz'},
        {'path': f'/Volumes/project/3025011.02/bids/derivatives/fsl/sub-{unique_subject_ids[0]:03d}/ses-mri04/func/sub-{unique_subject_ids[0]:03d}_ses-mri04_baseline_confounds.feat/stats/zstat1.nii.gz', 'colormap': 'hot', 'colormapNegative': 'winter', 'opacity': 0.4, 'cal_min': 2.3, 'cal_max': 10},
    ]))
except FileNotFoundError as e:
    print(f'No viewer, file missing: {e.filename}')

### Check movement

In [ ]:
joystick_confounds_dataframe, confounds_dataframe = load_joystick_with_confounds(unique_subject_ids[0], unique_sessions[2])

In [ ]:
mean_fd = confounds_dataframe['framewise_displacement'].mean()
if mean_fd >= mean_fd_cutoff:
    print(f'Mean FD of {mean_fd}mm => {mean_fd_cutoff}mm')

fd_proportion_exceeding_cutoff = (confounds_dataframe['framewise_displacement'] > mean_fd_cutoff).mean()
fd_proportion_cutoff = 0.2
if fd_proportion_exceeding_cutoff >= fd_proportion_cutoff:
    print(f'{fd_proportion_exceeding_cutoff:.1%} of volumes have FD > {mean_fd_cutoff}mm, >= {fd_proportion_cutoff:.0%}')

max_fd = confounds_dataframe['framewise_displacement'].max()
if max_fd >= max_fd_cutoff:
    print(f'Max FD of {max_fd}mm => {max_fd_cutoff}mm')

### Relate joystick movements to confound regressors

In [ ]:
# Correlate the joystick coordinate with FD and the six motion parameters, one value per volume
motion_columns = ['framewise_displacement', 'trans_x', 'trans_y', 'trans_z', 'rot_x', 'rot_y', 'rot_z']
correlations = joystick_confounds_dataframe[motion_columns].corrwith(joystick_confounds_dataframe['location_y'], method='spearman')
moderate_correlations = correlations[correlations.abs() >=  joystick_movement_correlation_threshold]

if not moderate_correlations.empty:
    print(f'Spearman\'s rho correlation between confound regressors and average joystick location within a volume.\n'
    f'Consists of {len(joystick_confounds_dataframe)} volumes with joystick samples.\n')
    print(moderate_correlations.round(3).rename('spearman rho').to_string())